# Session 4 Solution: Execute a Cloud Evaluation Run

This notebook uses Entra authentication (`DefaultAzureCredential`) and the live Foundry eval API shape via `get_openai_client().evals` methods.

In [1]:
from pathlib import Path
import sys
import time

candidates = [
    Path.cwd() / "solutions" / "src",
    Path.cwd().parent / "src",
    Path.cwd() / "src",
    Path.cwd().parent.parent / "solutions" / "src",
]
for path in candidates:
    if path.exists() and str(path) not in sys.path:
        sys.path.append(str(path))

from hackathon_solutions.config import load_config, validate_config
from hackathon_solutions.foundry_helpers import create_openai_client, load_jsonl

cfg = load_config()
validate_config(cfg)
openai_client = create_openai_client(cfg)
dataset_path = Path.cwd().parent / "data" / "session3_test_queries.jsonl"
if not dataset_path.exists():
    dataset_path = Path.cwd() / "solutions" / "data" / "session3_test_queries.jsonl"
rows = load_jsonl(dataset_path)
print("Rows loaded:", len(rows))

Rows loaded: 3


In [10]:
# do some basic validation to ensure the dataset is in the expected format before creating the evaluation
required_fields = {"query", "ground_truth"}
missing = [idx for idx, row in enumerate(rows, start=1) if not required_fields.issubset(row.keys())]
if missing:
    raise ValueError(f"Rows missing required fields {required_fields}: {missing[:5]}")

schema_properties = {key: {"type": "string"} for key in sorted({k for row in rows for k in row.keys()})}
item_schema = {
    "type": "object",
    "properties": schema_properties,
    "required": ["query", "ground_truth"],
}

evaluation = openai_client.evals.create(
    name=f"session4-eval-{int(time.time())}",
    data_source_config={
        "type": "custom",
        "item_schema": item_schema,
        "include_sample_schema": True,
    },
    # https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/azure-openai-graders
    testing_criteria=[
{
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {
            "deployment_name": cfg.model_deployment_name,
        },
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.ground_truth}}",
        },
        },
    ],
)

evaluation.id

'eval_72b4323948c644f6a10d55e453e54fb7'

In [ ]:
# # do some basic validation to ensure the dataset is in the expected format before creating the evaluation
# required_fields = {"query", "ground_truth"}
# missing = [idx for idx, row in enumerate(rows, start=1) if not required_fields.issubset(row.keys())]
# if missing:
#     raise ValueError(f"Rows missing required fields {required_fields}: {missing[:5]}")

# schema_properties = {key: {"type": "string"} for key in sorted({k for row in rows for k in row.keys()})}
# item_schema = {
#     "type": "object",
#     "properties": schema_properties,
#     "required": ["query", "ground_truth"],
# }

# evaluation = openai_client.evals.create(
#     name=f"session4-eval-{int(time.time())}",
#     data_source_config={
#         "type": "custom",
#         "item_schema": item_schema,
#         "include_sample_schema": True,
#     },
#     # https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/azure-openai-graders
#     testing_criteria=[
#         # {
#         #     "type": "score_model",
#         #     "name": "quality_score",
#         #     "model": cfg.model_deployment_name,
#         #     "pass_threshold": 0.85,
#         #     "input": [
#         #         {
#         #             "role": "system",
#         #             "content": "Score coherence on [0,1] for the assistant response. Return only the numeric score.",
#         #         },
#         #         {
#         #             "role": "user",
#         #             "content": "Query: {{item.query}}\nAnswer: {{sample.output_text}}",
#         #         },
#         #     ],
#         # },
#         # https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/azure-openai-graders#text-similarity-grader
#         {
#             "type": "text_similarity",
#             "name": "relevance",
#             "evaluation_metric": "cosine",
#             "input": "{{sample.output_text}}",
#             "reference": "{{item.ground_truth}}",
#             "pass_threshold": 0.85,
#         },
#     ],
# )

# evaluation.id

In [11]:
run = openai_client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"{evaluation.id}-run",
    data_source={
        "type": "responses",
        "model": cfg.model_deployment_name,
        "source": {
            "type": "file_content",
            "content": [{"item": row} for row in rows]
        },
        "input_messages": {
            "type": "template",
            "template": [{"role": "user", "content": "{{item.query}}"}]
        },
        "sampling_params": {"temperature": 0.0}
    }
)
while str(run.status).lower() in {"queued", "in_progress"}:
    time.sleep(8)
    run = openai_client.evals.runs.retrieve(run_id=run.id, eval_id=evaluation.id)

{
    "run_id": run.id,
    "status": run.status,
    "report_url": run.report_url,
    "result_counts": run.result_counts.model_dump() if run.result_counts else None
}

{'run_id': 'evalrun_1c92911a90454d54ac793638fd4477c4',
 'status': 'completed',
 'report_url': 'https://ai.azure.com/nextgen/r/TGmxZqOYSnWvA5cuV_bUiQ,SwedenAIML,,edwinfoundry0226,proj-default/build/evaluations/eval_72b4323948c644f6a10d55e453e54fb7/run/evalrun_1c92911a90454d54ac793638fd4477c4',
 'result_counts': {'errored': 0, 'failed': 0, 'passed': 3, 'total': 3}}

{'run_id': 'evalrun_1ec11b9c1d52422e8d7640dae3760ead',
 'status': 'completed',
 'report_url': 'https://ai.azure.com/nextgen/r/TGmxZqOYSnWvA5cuV_bUiQ,SwedenAIML,,edwinfoundry0226,proj-default/build/evaluations/eval_65055d3734cc468bbc2d741849a29dfd/run/evalrun_1ec11b9c1d52422e8d7640dae3760ead',
 'result_counts': {'errored': 0, 'failed': 0, 'passed': 3, 'total': 3}}